In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Widgets/Parameters

In [0]:
dbutils.widgets.text("catalog_name", "db_lakehouse_retail")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("storage_account", "retailstorageaccount19")
dbutils.widgets.text("raw_container", "ext-tables")
dbutils.widgets.text("checkpoint_container", "checkpoint")
dbutils.widgets.text("source_folder", "online_sales")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
storage_account = dbutils.widgets.get("storage_account")
raw_container = dbutils.widgets.get("raw_container")
checkpoint_container = dbutils.widgets.get("checkpoint_container")
source_folder = dbutils.widgets.get("source_folder")

In [0]:
source_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/{source_folder}/"
checkpoint_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_checkpoint/"
schema_location = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_schema/"
bad_records_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_bad_records/"

In [0]:
target_table = f"{catalog_name}.{bronze_schema}.online_orders_raw"

print(f"source_path : {source_path}")
print(f"checkpoint_location : {checkpoint_path}")
print(f"schema_location : {schema_location}")
print(f"bad_records_path : {bad_records_path}")


Read Data with autoloader

In [0]:
raw_stream_df = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "json")\
                    .option("cloudFiles.schemaLocation",schema_location)\
                    .option("cloudFiles.inferColumntypes", "true")\
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
                    .option("bad_records_path",bad_records_path)\
                    .option("multiline", "true")\
                    .load(source_path)

bronze_df = (raw_stream_df
             .withColumn("_source_file", col("_metadata.file_path"))
             .withColumn("_ingested_at", current_timestamp())
             .withColumn("_source_system", lit("online_ecommerce"))
             )


In [0]:
%python
query = bronze_df.writeStream.format('delta')\
            .option("checkpointLocation", checkpoint_path)\
            .option("mergeSchema", "true")\
            .trigger(once=True)\
            .toTable(target_table)

In [0]:
%sql
Select * from db_lakehouse_retail.bronze.online_orders_raw